# Detect DeepFake Images
## Train a MobileNetV3 binary classifier in PyTorch

In [ ]:
import pandas as pd
import numpy as np

## 1. Verify Dataset Structure

Inspect the folders under `data/Train`, `data/Validation`, and `data/Test` to ensure each has `Fake/` and `Real/` subdirectories and count the images.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data")
TRAIN_DIR = DATA_DIR / "Train"
VAL_DIR = DATA_DIR / "Validation"
TEST_DIR = DATA_DIR / "Test"

for split in ["Train", "Validation", "Test"]:
    split_dir = DATA_DIR / split
    print(f"{split}:")
    for label in ["Fake", "Real"]:
        label_dir = split_dir / label
        count = len(list(label_dir.glob("*"))) if label_dir.exists() else 0
        print(f"  {label}: {count} images")
print("Dataset structure verified")

## 2. Setup

Import libraries, configure the device (M3 Pro/MPS or CPU), set seeds, and ensure output directories exist.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)
import json
from pathlib import Path
from PIL import Image
import warnings

warnings.filterwarnings("ignore")

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

MODEL_DIR = Path("../models")
RESULTS_DIR = Path("../results")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model directory: {MODEL_DIR.resolve()}")
print(f"Results directory: {RESULTS_DIR.resolve()}")

## 3. Data Pipeline

Define `DeepfakeDataset` class and augmentations/transforms.

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.images = []
        self.labels = []
        for label, idx in {"Fake": 0, "Real": 1}.items():
            dirp = self.root_dir / label
            if dirp.exists():
                for p in sorted(dirp.glob("*")):
                    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
                        self.images.append(str(p))
                        self.labels.append(idx)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


IMG_SIZE = 256
BATCH_SIZE = 32
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.2, 0.2, 0.2),
        transforms.RandomHorizontalFlip(0.5),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)

val_test_transforms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)

print("Transforms and dataset class defined")

## 4. DataLoaders

Instantiate datasets and DataLoaders.

In [ ]:
train_dataset = DeepfakeDataset(TRAIN_DIR, train_transforms)
val_dataset = DeepfakeDataset(VAL_DIR, val_test_transforms)
test_dataset = DeepfakeDataset(TEST_DIR, val_test_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print(f"Train images: {len(train_dataset)}, batches: {len(train_loader)}")
print(f"Val images:   {len(val_dataset)}, batches: {len(val_loader)}")
print(f"Test images:  {len(test_dataset)}, batches: {len(test_loader)}")